# Series de tiempo: punto de arranque

Para Sebastián (y quien modele). Este notebook **no es el modelo final**: deja listo lo aburrido para que empieces a probar modelos con una base validada.

1. Carga las series que arma `python -m series_tiempo` (por hora y por día, con calendario colombiano).
2. Separa entrenamiento y prueba respetando el tiempo.
3. Define las métricas (MAE, RMSE y MASE).
4. Calcula **modelos de referencia** que cualquier modelo serio tiene que superar, y un primer SARIMAX con festivos.

### Antes de empezar
- Corre primero, con la VPN y el `.env` listos: `python -m series_tiempo` (genera el reporte y las series en `DATA_DIR\series_tiempo`). Lee el reporte `reporte_series.html`: ahí están las razones de cada decisión de abajo.
- **Lo que ves aquí son datos del HUN (agregados).** No pegues tablas ni gráficas en Claude, ChatGPT ni WhatsApp. Lo que sí puedes compartir con Claude: la salida de `python -m series_tiempo.resumen_seguro` y tus conclusiones en palabras ("el SARIMA le gana al naive semanal en algo así como un 10-20 %").
- **Antes de commitear: Clear All Outputs** y guarda. El hook bloquea notebooks con outputs.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from series_tiempo import cargar
from series_tiempo.datos import tramo_continuo

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)

s = cargar()                 # dict: 'horas', 'dias', 'grupos', 'semanas'
d, h = s["dias"], s["horas"]
d.filter(["llegadas", "festivo", "puente", "semana_santa", "carnaval", "completo"]).tail()

## 1. Entrenamiento y prueba

Solo usamos el **tramo continuo más largo de días completos** (los meses incompletos y los días cortados por el extracto ya vienen marcados con `completo = False`). La prueba son las **últimas 8 semanas** de ese tramo: así el modelo se evalúa en fechas que nunca vio, como pasaría en la vida real.

`H` es el horizonte: cuántos días pronosticamos de una vez. Cámbialo si el equipo decide otro (p. ej. 7 días para planear turnos).

In [ ]:
H = 56                                                   # días de prueba
tramo = d[tramo_continuo(d["completo"].astype(bool))]
y = tramo["llegadas"].astype(float).asfreq("D")
y_train, y_test = y.iloc[:-H], y.iloc[-H:]
print("entrenamiento:", y_train.index.min().date(), "a", y_train.index.max().date(), f"({len(y_train)} días)")
print("prueba:       ", y_test.index.min().date(), "a", y_test.index.max().date(), f"({len(y_test)} días)")

## 2. Métricas

- **MAE**: error absoluto medio, en llegadas por día. Fácil de explicar al HUN.
- **RMSE**: castiga más los errores grandes (los días que se desbordan).
- **MASE**: MAE dividido por el MAE del naive semanal *dentro del entrenamiento*. **MASE < 1 = le gana al naive semanal.** Es la que sirve para comparar modelos entre series distintas.

In [ ]:
def evaluar(nombre, y_true, y_pred, y_train=None, m=7):
    y_train = y_train_actual if y_train is None else y_train
    e = np.asarray(y_true) - np.asarray(y_pred)
    escala = np.mean(np.abs(y_train.to_numpy()[m:] - y_train.to_numpy()[:-m]))
    return {"modelo": nombre, "MAE": np.mean(np.abs(e)), "RMSE": np.sqrt(np.mean(e ** 2)),
            "MASE": np.mean(np.abs(e)) / escala}


y_train_actual = y_train
resultados = []
pronosticos = {}

## 3. Modelos de referencia (diarios)

- **Naive semanal**: cada día de la prueba = el mismo día de la última semana del entrenamiento. Es lo que haría alguien sin modelo.
- **Promedio por día de la semana**: el promedio del entrenamiento para cada día de la semana (últimas 12 semanas).

In [ ]:
ultima_semana = y_train.iloc[-7:].to_numpy()
pronosticos["naive semanal"] = pd.Series(np.resize(ultima_semana, H), index=y_test.index)

reciente = y_train.iloc[-84:]
prom_dia = reciente.groupby(reciente.index.dayofweek).mean()
pronosticos["promedio por día (12 sem)"] = pd.Series(prom_dia.reindex(y_test.index.dayofweek).to_numpy(), index=y_test.index)

for n in ["naive semanal", "promedio por día (12 sem)"]:
    resultados.append(evaluar(n, y_test, pronosticos[n]))
pd.DataFrame(resultados)

## 4. Un primer modelo: SARIMAX con calendario

SARIMA(1,0,1)(1,1,1)₇ con variables de festivos y temporadas. Los órdenes son un **punto de partida**, no el óptimo: elígelos con la ACF/PACF y las pruebas ADF/KPSS del reporte (secciones H e I), y compara por AIC y, sobre todo, por el error en la prueba.

Las exógenas salen de `series_diarias.parquet`. Si en el reporte (sección F) alguna categoría no se aleja de 1, puedes quitarla.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

EXOG = ["festivo", "despues_de_festivo", "puente", "semana_santa", "carnaval", "navidad_anio_nuevo"]
X = tramo[EXOG].astype(float).asfreq("D")
X = X.loc[:, X.iloc[:-H].std() > 0]                  # quita variables que no cambian en el entrenamiento
X_train, X_test = X.iloc[:-H], X.iloc[-H:]

sarimax = SARIMAX(y_train, exog=X_train, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
pronosticos["SARIMAX(1,0,1)(1,1,1)7 + calendario"] = sarimax.forecast(H, exog=X_test)
resultados.append(evaluar("SARIMAX(1,0,1)(1,1,1)7 + calendario", y_test, pronosticos["SARIMAX(1,0,1)(1,1,1)7 + calendario"]))
print(sarimax.summary().tables[1])
pd.DataFrame(resultados).sort_values("MASE")

In [ ]:
ax = y_test.plot(color="black", lw=2, label="real")
for n, p in pronosticos.items():
    p.plot(ax=ax, lw=1.3, label=n)
ax.set_title("Prueba: últimas semanas"); ax.set_ylabel("llegadas por día"); ax.legend(); plt.show()

res = sarimax.resid.iloc[8:]
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
res.plot(ax=axs[0], title="Residuos del SARIMAX (entrenamiento)")
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(res, lags=35, ax=axs[1], title="ACF de los residuos"); plt.tight_layout(); plt.show()

**Qué mirar:** los residuos deberían verse como ruido (sin patrón, ACF dentro de la banda). Si queda un pico en 7 o en 365, falta estacionalidad; si los errores grandes caen en fechas especiales, falta calendario.

## 5. Serie por hora: referencias

Lo mismo por hora, con horizonte de 2 semanas. El SARIMA con s=168 es muy lento; para la serie horaria suelen funcionar mejor MSTL + ARIMA, regresión con términos de Fourier (24 y 168) o un GLM Poisson/Binomial negativa (el reporte, sección J, dice cuál).

In [ ]:
Hh = 24 * 14
th = h[tramo_continuo(h["completo"].astype(bool))]["llegadas"].astype(float).asfreq("h")
yh_train, yh_test = th.iloc[:-Hh], th.iloc[-Hh:]
y_train_actual = yh_train                      # MASE horaria se escala con el naive de 168 h

ph = {"naive semanal (168 h)": pd.Series(np.resize(yh_train.iloc[-168:].to_numpy(), Hh), index=yh_test.index)}
rec = yh_train.iloc[-168 * 8:]
franja = rec.groupby([rec.index.dayofweek, rec.index.hour]).mean()
ph["promedio de la franja día×hora (8 sem)"] = pd.Series(
    franja.reindex(list(zip(yh_test.index.dayofweek, yh_test.index.hour))).to_numpy(), index=yh_test.index)
pd.DataFrame([evaluar(n, yh_test, p, yh_train, m=168) for n, p in ph.items()])

## 6. Siguientes pasos sugeridos

- **Validación por origen móvil** en vez de un solo corte: repetir el corte varias veces hacia atrás (p. ej. `sklearn.model_selection.TimeSeriesSplit`) y promediar el error. Un solo periodo de prueba puede engañar.
- Probar, según el reporte: **Prophet** con los festivos de Colombia, **MSTL + ARIMA**, **regresión con Fourier** (diaria, semanal y anual) y **GLM Binomial negativa** para la serie horaria.
- Si el reporte (sección L) muestra grupos con patrones distintos: pronóstico **por grupo** y sumar (jerárquico).
- Para la simulación y las alertas de saturación: además de llegadas, pronosticar **ingresos y altas de cama** (sección K).
- Guardar la comparación final (tabla de MAE/MASE por modelo) **en palabras o como agregado** en el documento; los pronósticos con fechas y valores reales no van al repo.

> Antes de cerrar: **Clear All Outputs** y guarda.